In [ ]:
# ==============================
# 1. IMPORT
# ==============================
import pandas as pd
import numpy as np
from scipy import stats



===== DESCRIPTIVE STATS =====
          price_egp    area_value  price_per_sqft
count  3.971300e+04  39713.000000    3.971300e+04
mean   7.989093e+06    218.484149    4.188689e+04
std    2.449255e+07    781.814044    1.242177e+05
min    1.300000e+03      1.000000    5.500000e+00
25%    6.500000e+04    130.000000    3.500000e+02
50%    6.000000e+05    175.000000    6.060606e+03
75%    9.115000e+06    245.000000    6.419753e+04
max    1.000000e+09  96600.000000    1.400000e+07

===== CORRELATION =====
                price_egp  area_value  price_per_sqft
price_egp        1.000000    0.411546        0.307708
area_value       0.411546    1.000000       -0.011971
price_per_sqft   0.307708   -0.011971        1.000000

Shapiro p-value: 7.304928426047481e-38

===== ANOVA TEST =====
ANOVA p-value: 1.927719718493923e-15

===== UNDERVALUATION =====

Top undervalued properties:
                                           location_full  price_per_sqft  \
10494  Hilton Cairo Nile Maadi, Athar El Nab

In [11]:

# ==============================
# 2. LOAD DATA
# ==============================
df = pd.read_csv("../data/processed/cleaned_data.csv")


In [12]:

# ==============================
# 3. USE CORRECT COLUMNS
# ==============================
price_col = "price_egp"
area_col = "area_value"
location_col = "location_full"


In [13]:

# ==============================
# 3. USE CORRECT COLUMNS
# ==============================
price_col = "price_egp"
area_col = "area_value"
location_col = "location_full"


In [14]:

# ==============================
# 4. CONVERT TO NUMERIC (VERY IMPORTANT)
# ==============================
df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
df[area_col] = pd.to_numeric(df[area_col], errors='coerce')

# Drop invalid rows
df = df.dropna(subset=[price_col, area_col])


In [15]:

# ==============================
# 5. PRICE PER SQFT
# ==============================
df['price_per_sqft'] = df[price_col] / df[area_col]

# ==============================
# 6. DESCRIPTIVE STATS
# ==============================
print("\n===== DESCRIPTIVE STATS =====")
print(df[[price_col, area_col, 'price_per_sqft']].describe())



===== DESCRIPTIVE STATS =====
          price_egp    area_value  price_per_sqft
count  3.971300e+04  39713.000000    3.971300e+04
mean   7.989093e+06    218.484149    4.188689e+04
std    2.449255e+07    781.814044    1.242177e+05
min    1.300000e+03      1.000000    5.500000e+00
25%    6.500000e+04    130.000000    3.500000e+02
50%    6.000000e+05    175.000000    6.060606e+03
75%    9.115000e+06    245.000000    6.419753e+04
max    1.000000e+09  96600.000000    1.400000e+07


In [16]:

# ==============================
# 7. CORRELATION
# ==============================
print("\n===== CORRELATION =====")
print(df[[price_col, area_col, 'price_per_sqft']].corr())

# ==============================
# 8. NORMALITY TEST
# ==============================
sample = df[price_col].sample(min(500, len(df)))
stat, p = stats.shapiro(sample)

print("\nShapiro p-value:", p)



===== CORRELATION =====
                price_egp  area_value  price_per_sqft
price_egp        1.000000    0.411546        0.307708
area_value       0.411546    1.000000       -0.011971
price_per_sqft   0.307708   -0.011971        1.000000

Shapiro p-value: 5.195247943952747e-33


In [17]:

# ==============================
# 9. ANOVA TEST
# ==============================
print("\n===== ANOVA TEST =====")

top_locs = df[location_col].value_counts().head(5).index
groups = [df[df[location_col] == loc]['price_per_sqft'] for loc in top_locs]

f, p_val = stats.f_oneway(*groups)

print("ANOVA p-value:", p_val)



===== ANOVA TEST =====
ANOVA p-value: 1.927719718493923e-15


In [18]:

# ==============================
# 10. UNDERVALUATION
# ==============================
print("\n===== UNDERVALUATION =====")

loc_avg = df.groupby(location_col)['price_per_sqft'].mean()

df['location_avg'] = df[location_col].map(loc_avg)
df['undervalued_index'] = (
    (df['location_avg'] - df['price_per_sqft']) / df['location_avg']
) * 100

undervalued = df.sort_values(by='undervalued_index', ascending=False)

print("\nTop undervalued properties:")
print(undervalued[[location_col, 'price_per_sqft', 'undervalued_index']].head(10))



===== UNDERVALUATION =====

Top undervalued properties:
                                           location_full  price_per_sqft  \
10494  Hilton Cairo Nile Maadi, Athar El Nabi, Hay Ma...    60869.565217   
16775  Hilton Cairo Nile Maadi, Athar El Nabi, Hay Ma...   140000.000000   
9275   Hilton Cairo Nile Maadi, Athar El Nabi, Hay Ma...   159800.000000   
16899  Hilton Cairo Nile Maadi, Athar El Nabi, Hay Ma...   169767.441860   
17301  Hilton Cairo Nile Maadi, Athar El Nabi, Hay Ma...   532666.666667   
4987   Jefaira Quayside, Jefaira, Ras Al Hekma, North...    67326.732673   
6140   Jefaira Quayside, Jefaira, Ras Al Hekma, North...    72028.985507   
6411   Jefaira Quayside, Jefaira, Ras Al Hekma, North...    72289.156627   
13187  Jefaira Quayside, Jefaira, Ras Al Hekma, North...   104897.023810   
12952  La Capitale, New Capital Compounds, New Capita...    36000.000000   

       undervaluation_score  
10494          2.449648e+06  
16775          2.370517e+06  
9275           2

In [19]:

# ==============================
# 11. SAVE FINAL
# ==============================
df.to_csv("../data/processed/final_dataset.csv", index=False)

print("\n✅ DONE — Statistical analysis completed")


✅ DONE — Statistical analysis completed
